# Lesson 01: Embeddings as Linear Algebra for Language Models

This notebook is a university-level introduction to embeddings, with math-first explanations and code that verifies each key claim.

Reference paper context: *Attention Is All You Need* (Vaswani et al., 2017).

## Learning objectives

After completing this notebook, you should be able to:
- Formalize embedding lookup as matrix multiplication with one-hot vectors
- Compute a forward pass from embeddings to token probabilities
- Derive and verify which embedding rows receive gradient updates
- Explain cosine geometry and why similarity emerges
- Understand sinusoidal positional encoding as Fourier-like features

## Notation

- Vocabulary size: $V$
- Embedding dimension: $d$
- Embedding matrix: $E \in \mathbb{R}^{V \times d}$
- Token id at position $t$: $i_t \in \{0, ..., V-1\}$
- One-hot vector for token $i$: $x_i \in \mathbb{R}^{V}$
- Embedded token vector: $e_i = x_i^\top E \in \mathbb{R}^{d}$

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(7)

## 1. Embedding lookup is linear algebra

Given one-hot $x_i$, embedding lookup is:

$$e_i = x_i^\top E$$

Because $x_i$ has a single 1 at index $i$, this selects row $i$ of $E$.

In [ ]:
vocab = ["the", "cat", "sat", "on", "mat", "dog"]
token_to_id = {tok: i for i, tok in enumerate(vocab)}
V = len(vocab)
d = 5

E = rng.normal(0, 0.2, size=(V, d))

tok = "cat"
i = token_to_id[tok]
x_i = np.zeros(V)
x_i[i] = 1.0

e_by_index = E[i]
e_by_matmul = x_i @ E

print(f"Token: {tok} (id={i})")
print("By indexing: ", e_by_index)
print("By matmul:   ", e_by_matmul)
print("Equal?", np.allclose(e_by_index, e_by_matmul))

For a sequence of token ids $(i_1, ..., i_T)$, stack one-hot rows into $X \in \mathbb{R}^{T \times V}$. Then:

$$H = XE \in \mathbb{R}^{T \times d}$$

So embedding lookup for a whole sequence is one matrix multiply.

In [ ]:
seq = ["the", "cat", "sat", "on", "the", "mat"]
ids = np.array([token_to_id[s] for s in seq])
T = len(ids)

X = np.zeros((T, V))
X[np.arange(T), ids] = 1.0
H = X @ E

print("ids:", ids)
print("H shape:", H.shape)
print("First two embedded vectors:
", H[:2])

## 2. From embeddings to probabilities

A minimal language-model head uses:

$$z_t = h_t W_{out} + b, \quad p_t = \text{softmax}(z_t)$$

where $W_{out} \in \mathbb{R}^{d \times V}$ and $z_t \in \mathbb{R}^{V}$ are logits.

In [ ]:
def softmax(x, axis=-1):
    x_shift = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x_shift)
    return e / np.sum(e, axis=axis, keepdims=True)

def cross_entropy_from_logits(logits, targets):
    probs = softmax(logits, axis=-1)
    p_true = probs[np.arange(len(targets)), targets]
    return -np.log(p_true + 1e-12).mean(), probs

W_out = rng.normal(0, 0.2, size=(d, V))
b = np.zeros(V)

# Predict next token for first T-1 positions
H_in = H[:-1]
targets = ids[1:]
logits = H_in @ W_out + b
loss, probs = cross_entropy_from_logits(logits, targets)

print("logits shape:", logits.shape)
print("loss:", round(float(loss), 6))
print("target token ids:", targets)

## 3. Which embedding rows get updated?

For one position with one-hot input $x_i$ and gradient wrt embedding output $g = \partial L / \partial e_i$, we have:

$$e_i = x_i^\top E \Rightarrow \frac{\partial L}{\partial E} = x_i g^\top$$

This outer product means only row $i$ is non-zero.
For a batch/sequence, gradients accumulate on rows corresponding to observed tokens.

In [ ]:
# Verify row-sparse gradient structure numerically for one token
i = token_to_id["cat"]
x_i = np.zeros(V)
x_i[i] = 1.0
g = rng.normal(size=d)

dE = np.outer(x_i, g)
row_norms = np.linalg.norm(dE, axis=1)

print("Token id updated:", i)
print("Row norms of dE:", row_norms)
print("Nonzero rows:", np.where(row_norms > 1e-12)[0])

In [ ]:
# Sequence-level accumulation: repeated tokens accumulate larger update norms
G = rng.normal(size=(T, d))  # pretend upstream gradients at each position
dE_seq = X.T @ G

for tok in vocab:
    idx = token_to_id[tok]
    count = int((ids == idx).sum())
    norm = np.linalg.norm(dE_seq[idx])
    print(f"{tok:>4s} | count={count} | grad_norm={norm:.4f}")

## 4. Geometry: cosine similarity

Cosine similarity between vectors $u, v$ is:

$$\cos(u,v) = \frac{u^\top v}{\|u\|\,\|v\|}$$

It measures angle, not magnitude. This is useful when vector norms vary during training.

In [ ]:
def cosine_similarity(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

# Synthetic semantic structure for demonstration
demo = {
    "cat": np.array([1.0, 1.0, 0.1]),
    "dog": np.array([0.9, 1.1, 0.1]),
    "car": np.array([-1.0, -0.8, 0.2]),
    "bus": np.array([-0.9, -1.0, 0.2]),
}

pairs = [("cat", "dog"), ("cat", "car"), ("car", "bus")]
for a, b in pairs:
    print(f"cos({a},{b}) = {cosine_similarity(demo[a], demo[b]):+.4f}")

## 5. Positional encoding as Fourier features

Transformer input is often:

$$x_t = e_t + p_t$$

where sinusoidal $p_t$ uses paired dimensions:

$$p_{t,2k} = \sin(\omega_k t), \quad p_{t,2k+1} = \cos(\omega_k t)$$

with frequencies $\omega_k = 10000^{-2k/d}$.

For each frequency pair, dot products depend on relative distance:

$$\sin(\omega t)\sin(\omega s) + \cos(\omega t)\cos(\omega s) = \cos(\omega (t-s))$$

So relative position information is linearly recoverable from these features.

In [ ]:
def sinusoidal_positional_encoding(max_len, d_model):
    pe = np.zeros((max_len, d_model))
    pos = np.arange(max_len)[:, None]
    freq = np.exp(np.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
    pe[:, 0::2] = np.sin(pos * freq)
    pe[:, 1::2] = np.cos(pos * freq)
    return pe

pe = sinusoidal_positional_encoding(max_len=16, d_model=8)

t, s = 9, 5
lhs = pe[t] @ pe[s]
print(f"Dot(pe[{t}], pe[{s}]) = {lhs:.6f}")

# Check dependence on relative offset for one frequency pair
w0 = np.exp(0 * (-np.log(10000.0) / 8))
rhs_pair0 = np.cos(w0 * (t - s))
print(f"Single-pair relative term cos(w0*(t-s)) = {rhs_pair0:.6f}")

## Exercises (university track)

1. Prove that for sequence embedding matrix $H = XE$, the gradient is $\partial L/\partial E = X^\top(\partial L/\partial H)$.
2. Show that tied embeddings ($W_{out} = E^\top$) reduce parameter count; compute savings for your chosen $V,d$.
3. For sinusoidal encoding, derive the full expression for $p_t^\top p_s$ as a sum of cosines over frequencies.
4. Implement a tiny SGD loop that learns embeddings on a toy corpus and track cosine neighborhoods over epochs.

## Next notebook

Continue with [02_attention_mechanism.ipynb](./02_attention_mechanism.ipynb) for formal derivation of scaled dot-product attention and masking.